# How to compare multiple ML models
In this notebook, you will build a simple and flexible machine learning pipeline that makes it easy to switch between different models. Pipelines help tidy up your code, reduce repetition, and ensure each step in the process is applied consistently. Whether you are trying out a decision tree, a random forest, or a K nearest neighbours model, this approach will let you test them all without needing to rewrite everything from scratch. It is a practical way to keep your experiments organised and your workflow smooth as you compare model performance and find the best fit for your data.

## 1.&nbsp;Preprocessor
Before feeding data into a machine learning model, it is important to make sure it is in the right shape and format. This often means handling missing values, scaling numerical features, and encoding categorical ones. A preprocessor takes care of these steps in a structured and consistent way, so you do not have to repeat the same code each time. It also helps avoid mistakes by keeping the transformations the same for all models. Using a preprocessor ensures your data is properly prepared for any model you choose to use.

In [ ]:
# from numpy._core import numeric
# num_columns= X.select_types(include="number").columns
# cat_columns= X.select_types(exclude="number").columns


# numeric_subpipe = make_pipeline(
#     SimpleImputer(strategy="mean")
# )

# categorical_subpipe= make_pipeline(
#     SimpleImputer(strategy="constant", fill_value="N_A")
#     OneHotEncoder(sparse_output=False, handle_unknown="ignore")
# )

# preprocessor = make_column_transformer(
#     (numeric_subpipe, num_columns),
#     (categorical_subpipe, cat_columns)
# )

# full_pipeline_dt = make_pipeline(
#     preprocessor,
#     DecisionTreeClassifier()
# )

# full_pipeline_knn = make_pipeline(
#     preprocessor,
    # MinMaxScaler(),
#     KNeighborsClassifier()
# )

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn import set_config
set_config(transform_output='pandas')

# -----------------------------
# Data Loading
# -----------------------------
url = "https://drive.google.com/file/d/1g3uhw_y3tboRm2eYDPfUzXXsw8IOYDCy/view?usp=sharing"
download_url = 'https://drive.google.com/uc?export=download&id=' + url.split('/')[-2]
data = pd.read_csv(download_url)

# -----------------------------
# Feature and Target Separation
# -----------------------------
# Drop columns that are not useful for prediction and separate the target variable.
X = data.drop(columns=["PassengerId", "Name", "Ticket"])
y = X.pop("Survived")

# -----------------------------
# Feature Engineering
# -----------------------------
# Extract the first character of the 'Cabin' column to capture deck information.
X["Cabin"] = X["Cabin"].str[0]

# -----------------------------
# Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=123)

# -----------------------------
# Preprocessing Pipelines
# -----------------------------
# Identify numerical and categorical features
num_features = X.select_dtypes(include="number").columns
cat_features = X.select_dtypes(exclude="number").columns

# Numerical pipeline: Impute missing values using the mean.
num_pipe = make_pipeline(SimpleImputer(strategy="mean"))

# Categorical preprocessing:
# - Impute missing categorical values with a constant 'N_A'
cat_imputer = SimpleImputer(strategy="constant", fill_value="N_A")

# For ordinal encoding, specify the feature and its explicit order.
ordinal_features = ["Cabin"]
cabin_categories = ["N_A", "G", "F", "E", "D", "C", "B", "A", "T"]
ord_encoder = OrdinalEncoder(categories=[cabin_categories])

# Identify nominal categorical features (those that will be one-hot encoded)
nominal_features = list(set(cat_features) - set(ordinal_features))
oh_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Combine ordinal and nominal encoders using a column transformer.
cat_encoder = make_column_transformer(
    (ord_encoder, ordinal_features),
    (oh_encoder, nominal_features)
)

# Create a categorical pipeline: imputation followed by encoding.
cat_pipe = make_pipeline(cat_imputer, cat_encoder)

# Combine numerical and categorical pipelines into a full preprocessor.
preprocessor = make_column_transformer(
    (num_pipe, num_features),
    (cat_pipe, cat_features)
)

preprocessor

ColumnTransformer(transformers=[('pipeline-1',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer())]),
                                 Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                ('pipeline-2',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='N_A',
                                                                strategy='constant')),
                                                 ('columntransformer',
                                                  ColumnTransformer(transformers=[('ordinalencoder',
                                                                                   OrdinalEncoder(categories=[['N_A',
                                                                                                               'G',
                                                                                                               'F',
                                                                                                               'E',
                                                                                                               'D',
                                                                                                               'C',
                                                                                                               'B',
                                                                                                               'A',
                                                                                                               'T']]),
                                                                                   ['Cabin']),
                                                                                  ('onehotencoder',
                                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                                 sparse_output=False),
                                                                                   ['Embarked',
                                                                                    'Sex'])]))]),
                                 Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])

## 2.&nbsp;Decision Tree
Now that the preprocessor is set up, we can connect it to a machine learning model to build a complete pipeline. In this example, we will use a decision tree classifier, but the same structure can be used with any model. By combining the preprocessor and model into a single pipeline, we ensure that all the steps, from cleaning and transforming the data to fitting the model, are applied in the correct order. This also makes it easier to tune hyperparameters, since we can search across both preprocessing and modelling steps in one go.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

# Define a function to perform a grid search, which helps to avoid duplicating code for different models
def run_grid_search(model, param_grid, X_train, y_train, preprocessor, cv=5, verbose=1):
    # Create a pipeline that first applies the data preprocessing steps, then fits the model
    pipe = make_pipeline(preprocessor, model)

    # GridSearchCV will test all possible combinations of parameters defined in 'param_grid'
    grid_search = GridSearchCV(pipe, param_grid, cv=cv, verbose=verbose)

    # Fit the model on the training data with the various parameter combinations
    grid_search.fit(X_train, y_train)

    # Return the trained GridSearchCV object which holds the best parameters and model
    return grid_search

# Define a dictionary of hyperparameters to tune for the decision tree model
dt_param_grid = {
    "columntransformer__pipeline-1__simpleimputer__strategy": ["mean", "median"],
    "decisiontreeclassifier__max_depth": range(2, 14, 2),
    "decisiontreeclassifier__min_samples_leaf": range(3, 12, 2)
}

# Run the grid search for the DecisionTreeClassifier using the specified parameters
dt_search = run_grid_search(
    DecisionTreeClassifier(random_state=123),
    dt_param_grid,
    X_train,
    y_train,
    preprocessor
)

# Display the process
dt_search

Fitting 5 folds for each of 60 candidates, totalling 300 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('pipeline-1',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer())]),
                                                                         Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                                                        ('pipeline-2',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(fill_value='N_A',
                                                                                                        strategy='constant')),
                                                                                         ('columntransformer',
                                                                                          Col...
                                                                                                                           ['Sex',
                                                                                                                            'Embarked'])]))]),
                                                                         Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])),
                                       ('decisiontreeclassifier',
                                        DecisionTreeClassifier(random_state=123))]),
             param_grid={'columntransformer__pipeline-1__simpleimputer__strategy': ['mean',
                                                                                    'median'],
                         'decisiontreeclassifier__max_depth': range(2, 14, 2),
                         'decisiontreeclassifier__min_samples_leaf': range(3, 12, 2)},
             verbose=1)

Once we have trained and tuned our model, the next step is to check how well it performs. Here, we collect a set of evaluation scores for the decision tree and store them in a DataFrame. This gives us a clear summary of key metrics like accuracy, precision, recall, and F1 score, along with others that help us assess the model from different angles. The structure is designed so we can easily add results from other models later, making it simple to compare performance side by side and choose the most suitable approach for our task.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score
)

# Function to get the scores for our model(s)
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    scores = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Specificity": recall_score(y_test, y_pred, pos_label=0),
        "F1 Score": f1_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Cohen's Kappa": cohen_kappa_score(y_test, y_pred)
    }
    return scores

# Create an empty DataFrame to store model evaluation results
model_scores_df = pd.DataFrame(columns=[
    "Model", "Accuracy", "Recall", "Precision",
    "Specificity", "F1 Score", "Balanced Accuracy", "Cohen's Kappa"
])

# Evaluate the Decision Tree model
dt_scores = evaluate_model(dt_search, X_test, y_test)
dt_scores["Model"] = "Decision Tree"

# Convert the dictionary to a Series matching the DataFrame columns, then assign as a new row
model_scores_df.loc[len(model_scores_df)] = pd.Series(dt_scores, index=model_scores_df.columns)

# Display the DataFrame
model_scores_df

,Model,Accuracy,Recall,Precision,Specificity,F1 Score,Balanced Accuracy,Cohen's Kappa
0,Decision Tree,0.843575,0.815385,0.768116,0.859649,0.791045,0.837517,0.666223


## 3.&nbsp;KNN
Now that we have results for the decision tree, we can try a second model using the same setup. This time we are testing a K nearest neighbours classifier. The structure is almost identical to what we used before, same preprocessor, same grid search function, just with a different model and parameter grid. This shows how easy it is to test different approaches when your pipeline is set up in a consistent way.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Define the hyperparameter grid to be searched by the grid search
knn_param_grid = {
    "kneighborsclassifier__n_neighbors": range(1, 11),
    "kneighborsclassifier__weights": ["uniform", "distance"]
}

# Run a grid search to find the optimal combination of hyperparameters
knn_search = run_grid_search(
    KNeighborsClassifier(),
    knn_param_grid,
    X_train,
    y_train,
    preprocessor
)

# Display the grid search results
knn_search

Fitting 5 folds for each of 20 candidates, totalling 100 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('pipeline-1',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer())]),
                                                                         Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                                                        ('pipeline-2',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(fill_value='N_A',
                                                                                                        strategy='constant')),
                                                                                         ('columntransformer',
                                                                                          Col...
                                                                                                                                                       'A',
                                                                                                                                                       'T']]),
                                                                                                                           ['Cabin']),
                                                                                                                          ('onehotencoder',
                                                                                                                           OneHotEncoder(handle_unknown='ignore',
                                                                                                                                         sparse_output=False),
                                                                                                                           ['Sex',
                                                                                                                            'Embarked'])]))]),
                                                                         Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])),
                                       ('kneighborsclassifier',
                                        KNeighborsClassifier())]),
             param_grid={'kneighborsclassifier__n_neighbors': range(1, 11),
                         'kneighborsclassifier__weights': ['uniform',
                                                           'distance']},
             verbose=1)

With the K nearest neighbours model trained and tuned, we can now evaluate its performance and add the results to our scores DataFrame. Since we are using the same evaluation function and column structure, the process is quick and consistent.

In [ ]:
# Evaluate the K-Nearest Neighbours (KNN) model using the testing dataset and obtain performance metrics
knn_scores = evaluate_model(knn_search, X_test, y_test)

# Label the metrics to indicate they belong to the KNN model
knn_scores["Model"] = "KNN"

# Append the KNN metrics as a new row to the existing DataFrame of model scores
model_scores_df.loc[len(model_scores_df)] = pd.Series(knn_scores, index=model_scores_df.columns)

# Display the updated DataFrame containing all model performance metrics
model_scores_df

,Model,Accuracy,Recall,Precision,Specificity,F1 Score,Balanced Accuracy,Cohen's Kappa
0,Decision Tree,0.843575,0.815385,0.768116,0.859649,0.791045,0.837517,0.666223
1,KNN,0.73743,0.661538,0.632353,0.780702,0.646617,0.72112,0.437897


## 4.&nbsp;Challenge
Now that you have seen how to set up a pipeline and test different classifiers, it's time to put it into practice. Use the same approach to try out several models on your housing dataset and see how they compare. There is a page on the LMS that introduces a range of classifiers to get you started, but if you are feeling curious and want to explore even more options, the Scikit-learn documentation is a great place to look.